# Notebook 05 — Inference demo

**What this notebook does:**
1. Load a trained GNN (prefers pretrained / random split)
2. Predict W / L / E3 on a few test molecules
3. Draw colored molecule images under `outputs/figures/`

All drawing helpers are inlined below.


## 0. Paths

In [ ]:
# Project root = parent of notebooks/ (or cwd if already in Yashi/)
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "Protac_4.0_database_files_downloaded"
PROCESSED = ROOT / "data" / "processed"
OUT = ROOT / "outputs"
PROCESSED.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT)


## 1. Helper functions — model + draw

In [ ]:
from __future__ import annotations

import io
import math
import random
from collections import Counter, deque
from dataclasses import dataclass
from typing import Iterable, Optional

import networkx as nx
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Draw, rdMolDescriptors
from rdkit.Chem.Draw import rdMolDraw2D

RDLogger.DisableLog("rdApp.*")

WARHEAD, LINKER, E3 = 0, 1, 2
CLASS_NAMES = {WARHEAD: "warhead", LINKER: "linker", E3: "e3_ligand"}
CLASS_COLORS = {
    WARHEAD: (1.0, 0.35, 0.35),   # pink
    LINKER:  (0.30, 0.55, 1.0),   # blue
    E3:      (0.30, 0.85, 0.35),  # green
}

# =============================================================================
# 1. Weak labelling  (dictionary + reassembly filter)
# =============================================================================

@dataclass
class RefFragment:
    smiles: str
    mol: Chem.Mol
    n_atoms: int


def prepare_reference_library(
    smiles_list: Iterable[str], min_atoms: int = 5, max_atoms: int = 60
) -> list[RefFragment]:
    """Parse + filter reference SMILES; sort largest-first for greedy match."""
    refs: list[RefFragment] = []
    for smi in smiles_list:
        if not isinstance(smi, str) or not smi:
            continue
        m = Chem.MolFromSmiles(smi)
        if m is None:
            continue
        n = m.GetNumHeavyAtoms()
        if not (min_atoms <= n <= max_atoms):
            continue
        refs.append(RefFragment(Chem.MolToSmiles(m), m, n))
    refs.sort(key=lambda r: r.n_atoms, reverse=True)
    return refs


def _best_match(mol: Chem.Mol, refs: list[RefFragment],
                forbidden: Optional[set[int]] = None) -> Optional[tuple[set[int], str]]:
    """Return (atoms_of_best_match, ref_smiles) or None."""
    forbidden = forbidden or set()
    n_target = mol.GetNumAtoms()
    for ref in refs:
        if ref.n_atoms > n_target:
            continue
        matches = mol.GetSubstructMatches(ref.mol, uniquify=True, useChirality=False)
        for match in matches:
            atoms = set(match)
            if atoms.isdisjoint(forbidden):
                return atoms, ref.smiles
    return None


def _boundary_bonds(mol: Chem.Mol, labels: list[int]) -> list[int]:
    return [b.GetIdx() for b in mol.GetBonds()
            if labels[b.GetBeginAtomIdx()] != labels[b.GetEndAtomIdx()]]


def reassembles(mol: Chem.Mol, labels: list[int]) -> tuple[bool, int]:
    """Cut molecule at boundary bonds; return (success, n_fragments).

    Reassembly succeeds when cutting produces exactly 3 chemically valid
    fragments and no atoms are lost.  This is the Ribes-style validation
    of a splitting, applied to *any* atom-level labelling.
    """
    b_idx = _boundary_bonds(mol, labels)
    if not b_idx:
        return False, 1
    frag_mol = Chem.FragmentOnBonds(mol, b_idx, addDummies=True)
    frags = Chem.GetMolFrags(frag_mol, asMols=True, sanitizeFrags=False)
    if len(frags) != 3:
        return False, len(frags)
    atoms_seen = 0
    for f in frags:
        try:
            Chem.SanitizeMol(f)
        except Exception:
            return False, len(frags)
        atoms_seen += sum(1 for a in f.GetAtoms() if a.GetAtomicNum() != 0)
    return (atoms_seen == mol.GetNumAtoms()), len(frags)


def label_protac(smiles: str, wh_refs: list[RefFragment], e3_refs: list[RefFragment],
                 require_reassembly: bool = True) -> Optional[dict]:
    """Weak-label one PROTAC. Return dict or None on failure."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    n = mol.GetNumAtoms()
    if not (15 <= n <= 120):
        return None
    wh = _best_match(mol, wh_refs)
    if wh is None:
        return None
    e3 = _best_match(mol, e3_refs, forbidden=wh[0])
    if e3 is None:
        return None
    labels = [LINKER] * n
    for a in wh[0]:
        labels[a] = WARHEAD
    for a in e3[0]:
        labels[a] = E3
    if LINKER not in labels:
        return None
    if require_reassembly:
        ok, _ = reassembles(mol, labels)
        if not ok:
            return None
    return {
        "smiles": Chem.MolToSmiles(mol),
        "labels": labels,
        "wh_ref": wh[1],
        "e3_ref": e3[1],
        "n_warhead": len(wh[0]),
        "n_linker": labels.count(LINKER),
        "n_e3": len(e3[0]),
    }

# =============================================================================
# 3. Graph construction  (no PyTorch Geometric)
# =============================================================================

ATOM_TYPES = ["C", "N", "O", "S", "F", "Cl", "Br", "I", "P", "B", "Si", "Se", "Other"]
HYBRIDS = [Chem.HybridizationType.SP, Chem.HybridizationType.SP2,
           Chem.HybridizationType.SP3, Chem.HybridizationType.SP3D,
           Chem.HybridizationType.SP3D2]
BOND_TYPES = [Chem.BondType.SINGLE, Chem.BondType.DOUBLE,
              Chem.BondType.TRIPLE, Chem.BondType.AROMATIC]


def _oh(v, choices):
    x = [0.0] * (len(choices) + 1)
    try:
        x[choices.index(v)] = 1.0
    except ValueError:
        x[-1] = 1.0
    return x


def atom_features(a: Chem.Atom) -> list[float]:
    return (_oh(a.GetSymbol(), ATOM_TYPES) + _oh(a.GetHybridization(), HYBRIDS)
            + [float(a.GetDegree()), float(a.GetFormalCharge()),
               float(a.GetTotalNumHs()), float(a.GetIsAromatic()),
               float(a.IsInRing())])


def bond_features(b: Chem.Bond) -> list[float]:
    return _oh(b.GetBondType(), BOND_TYPES) + [float(b.GetIsConjugated()),
                                               float(b.IsInRing())]


ATOM_FEATURE_DIM = len(atom_features(Chem.MolFromSmiles("C").GetAtomWithIdx(0)))
BOND_FEATURE_DIM = len(bond_features(Chem.MolFromSmiles("CC").GetBondWithIdx(0)))


def smiles_to_graph(smiles: str, labels: Optional[list[int]] = None) -> Optional[dict]:
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    n = mol.GetNumAtoms()
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float32)
    src, dst, eattr = [], [], []
    for b in mol.GetBonds():
        i, j = b.GetBeginAtomIdx(), b.GetEndAtomIdx()
        f = bond_features(b)
        src += [i, j]; dst += [j, i]; eattr += [f, f]
    ei = torch.tensor([src, dst], dtype=torch.long) if src else torch.zeros(2, 0, dtype=torch.long)
    ea = torch.tensor(eattr, dtype=torch.float32) if eattr else torch.zeros(0, BOND_FEATURE_DIM)
    g = {"x": x, "edge_index": ei, "edge_attr": ea, "n_atoms": n, "smiles": smiles}
    if labels is not None:
        if len(labels) != n:
            return None
        g["y"] = torch.tensor(labels, dtype=torch.long)
    return g


def collate(graphs: list[dict]) -> dict:
    """Block-diagonal batch. `batch` maps each atom to its molecule id."""
    xs, eis, eas, ys, batch = [], [], [], [], []
    off = 0
    for i, g in enumerate(graphs):
        xs.append(g["x"]); eis.append(g["edge_index"] + off); eas.append(g["edge_attr"])
        batch.append(torch.full((g["n_atoms"],), i, dtype=torch.long))
        if "y" in g:
            ys.append(g["y"])
        off += g["n_atoms"]
    out = {
        "x": torch.cat(xs, 0),
        "edge_index": torch.cat(eis, 1),
        "edge_attr": torch.cat(eas, 0),
        "batch": torch.cat(batch, 0),
        "n_graphs": len(graphs),
    }
    if ys:
        out["y"] = torch.cat(ys, 0)
    return out

# =============================================================================
# 4. Model + losses
# =============================================================================

def scatter_sum(src: torch.Tensor, index: torch.Tensor, n: int) -> torch.Tensor:
    out = torch.zeros(n, src.size(-1), device=src.device, dtype=src.dtype)
    out.index_add_(0, index, src)
    return out


def scatter_mean(src: torch.Tensor, index: torch.Tensor, n: int) -> torch.Tensor:
    tot = scatter_sum(src, index, n)
    cnt = scatter_sum(torch.ones_like(src[:, :1]), index, n).clamp_min(1.0)
    return tot / cnt


class GINLayer(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.eps = nn.Parameter(torch.zeros(1))
        self.mlp = nn.Sequential(nn.Linear(dim, dim), nn.ReLU(), nn.Linear(dim, dim))
        self.bn = nn.BatchNorm1d(dim)

    def forward(self, x, edge_index):
        src, dst = edge_index[0], edge_index[1]
        agg = scatter_sum(x[src], dst, n=x.size(0))
        return self.bn(self.mlp((1.0 + self.eps) * x + agg))


class ProtacSegGNN(nn.Module):
    """Shared GIN encoder + atom head + bond head + AttrMask pretrain head.

    AttrMask (Hu et al. 2020; Mole-BERT family) masks atom *features* and
    asks the encoder to recover atom type — a self-supervised pretraining
    task that does not need W/L/E3 labels.
    """

    def __init__(self, atom_dim: int, hidden: int = 64, n_layers: int = 4,
                 dropout: float = 0.1, n_atom_types: int = len(ATOM_TYPES) + 1):
        super().__init__()
        self.encoder = nn.Linear(atom_dim, hidden)
        self.layers = nn.ModuleList([GINLayer(hidden) for _ in range(n_layers)])
        self.dropout = nn.Dropout(dropout)
        self.atom_head = nn.Linear(hidden, 3)
        self.bond_head = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                       nn.Linear(hidden, 2))
        self.frag_proj = nn.Linear(hidden, hidden)
        self.attrmask_head = nn.Linear(hidden, n_atom_types)

    def encode(self, x, edge_index):
        h = F.relu(self.encoder(x))
        for layer in self.layers:
            h = F.relu(layer(h, edge_index))
            h = self.dropout(h)
        return h

    def forward(self, x, edge_index):
        h = self.encode(x, edge_index)
        atom_logits = self.atom_head(h)
        src, dst = edge_index[0], edge_index[1]
        bond_logits = self.bond_head(torch.cat([h[src], h[dst]], dim=-1))
        return atom_logits, bond_logits, h

    def forward_attrmask(self, x, edge_index):
        """Return AttrMask logits over atom types (used during pretraining)."""
        h = self.encode(x, edge_index)
        return self.attrmask_head(h)

    def fragment_embeddings(self, h: torch.Tensor, atom_labels: torch.Tensor,
                            batch: torch.Tensor, n_graphs: int) -> torch.Tensor:
        """Mean-pool atom vectors per (graph, class) -> shape (n_graphs, 3, hidden)."""
        proj = self.frag_proj(h)
        out = torch.zeros(n_graphs, 3, proj.size(-1), device=h.device)
        for c in range(3):
            mask = (atom_labels == c).float().unsqueeze(-1)
            num = scatter_sum(proj * mask, batch, n=n_graphs)
            den = scatter_sum(mask, batch, n=n_graphs).clamp_min(1.0)
            out[:, c] = num / den
        return out


# ---- losses ---------------------------------------------------------------

def bond_boundary_labels(y: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
    return (y[edge_index[0]] != y[edge_index[1]]).long()


def bond_boundary_loss(bond_logits, y, edge_index):
    return F.cross_entropy(bond_logits, bond_boundary_labels(y, edge_index))


def fragment_smoothness_loss(atom_logits, y, edge_index):
    src, dst = edge_index[0], edge_index[1]
    same = (y[src] == y[dst]).float().unsqueeze(-1)
    if same.sum() == 0:
        return atom_logits.new_zeros(())
    p = F.softmax(atom_logits, dim=-1)
    diff = (p[src] - p[dst]).pow(2).sum(-1, keepdim=True)
    return (same * diff).sum() / same.sum().clamp_min(1.0)


def linker_attachment_loss(atom_logits: torch.Tensor, edge_index: torch.Tensor,
                           batch: torch.Tensor, n_graphs: int) -> torch.Tensor:
    """Soft-reconstruction proxy: linker should have ~2 boundary bonds.

    For each undirected bond (i,j) we compute
        p_boundary_linker(i,j) = p_L(i)(1-p_L(j)) + p_L(j)(1-p_L(i))
    Summing per molecule should ideally equal 2.  We normalise by a soft
    scale so the loss stays O(1) regardless of molecule size.
    """
    p = F.softmax(atom_logits, dim=-1)
    pL_i = p[edge_index[0], LINKER]
    pL_j = p[edge_index[1], LINKER]
    edge_boundary = pL_i * (1 - pL_j) + pL_j * (1 - pL_i)
    edge_batch = batch[edge_index[0]]
    per_mol = scatter_sum(edge_boundary.unsqueeze(-1), edge_batch, n=n_graphs).squeeze(-1) / 2.0
    # relative L2 around target=2 keeps magnitude stable
    return ((per_mol - 2.0) / (per_mol.detach() + 2.0)).pow(2).mean()


def class_presence_loss(atom_logits: torch.Tensor, batch: torch.Tensor,
                        n_graphs: int) -> torch.Tensor:
    """Every PROTAC should have non-trivial soft mass on all 3 classes.

    Soft presence per class ≈ 1 - ∏_atoms (1 - p_atom,class).
    Pushing presence toward 1 keeps W, L and E3 alive — critical for
    reassembly (a missing class cannot reconstruct 3 fragments).
    """
    p = F.softmax(atom_logits, dim=-1)  # (N, 3)
    presence = []
    for c in range(3):
        log1mp = torch.log((1.0 - p[:, c]).clamp(min=1e-6))
        s = scatter_sum(log1mp.unsqueeze(-1), batch, n=n_graphs).squeeze(-1)
        presence.append(1.0 - torch.exp(s))
    presence = torch.stack(presence, dim=-1)  # (G, 3)
    return F.mse_loss(presence, torch.ones_like(presence))


def soft_reconstruction_loss(atom_logits: torch.Tensor, edge_index: torch.Tensor,
                             batch: torch.Tensor, n_graphs: int) -> torch.Tensor:
    """Differentiable reconstruction objective used *during training*.

    Combines two proxies for the non-differentiable reassembly check:
      1. linker-attachment: ~2 linker boundary bonds (topology of a PROTAC)
      2. class-presence: all three classes must survive soft predictions

    This is the novelty-to-complexity sweet spot: reconstruction is part of
    the training methodology, not only an evaluation metric.
    """
    L_attach = linker_attachment_loss(atom_logits, edge_index, batch, n_graphs)
    L_pres = class_presence_loss(atom_logits, batch, n_graphs)
    return L_attach + 0.5 * L_pres


# ---- AttrMask pretraining -------------------------------------------------

def atom_type_index(x: torch.Tensor) -> torch.Tensor:
    """Recover atom-type index from one-hot prefix of feature vector."""
    # first len(ATOM_TYPES)+1 dims are the atom-type one-hot (see atom_features)
    n_types = len(ATOM_TYPES) + 1
    return x[:, :n_types].argmax(dim=-1)


def pretrain_attrmask_epoch(model: ProtacSegGNN, graphs: list[dict],
                            indices: list[int], optimizer,
                            batch_size: int = 32, mask_rate: float = 0.15) -> float:
    """One AttrMask epoch: mask atom features, predict atom types."""
    model.train()
    total_loss, n_masked = 0.0, 0
    for b in iterate_minibatches(graphs, indices, batch_size, shuffle=True):
        x = b["x"].clone()
        y_type = atom_type_index(x)
        n = x.size(0)
        n_mask = max(1, int(mask_rate * n))
        mask_idx = torch.randperm(n)[:n_mask]
        x[mask_idx] = 0.0  # zero-out features of masked atoms
        optimizer.zero_grad()
        logits = model.forward_attrmask(x, b["edge_index"])
        loss = F.cross_entropy(logits[mask_idx], y_type[mask_idx])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_loss += loss.item() * n_mask
        n_masked += n_mask
    return total_loss / max(n_masked, 1)


def pretrain_encoder(model: ProtacSegGNN, graphs: list[dict], indices: list[int],
                     epochs: int = 5, lr: float = 1e-3, batch_size: int = 32) -> list[float]:
    """Self-supervised AttrMask pretraining of the shared GIN encoder."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    hist = []
    for ep in range(1, epochs + 1):
        loss = pretrain_attrmask_epoch(model, graphs, indices, opt, batch_size=batch_size)
        hist.append(loss)
        print(f'  AttrMask ep {ep:02d}  loss={loss:.3f}')
    return hist

# =============================================================================
# 5. Train + evaluate
# =============================================================================

def iterate_minibatches(graphs: list[dict], indices: list[int], batch_size: int,
                        shuffle: bool = True):
    order = list(indices)
    if shuffle:
        random.shuffle(order)
    for i in range(0, len(order), batch_size):
        yield collate([graphs[j] for j in order[i:i + batch_size]])


def compute_class_weights(graphs: list[dict], indices: list[int],
                          linker_boost: float = 1.5) -> torch.Tensor:
    """Inverse-frequency class weights, normalised to mean 1.

    `linker_boost` extra multiplier on the linker class because it is
    the smallest and the hardest to recover (misclassifying it destroys
    reassembly).  Empirically 2.0 stops the model from collapsing to a
    'no linker' solution on unseen-E3 splits.
    """
    counts = torch.zeros(3)
    for i in indices:
        y = graphs[i]["y"]
        for c in range(3):
            counts[c] += (y == c).sum().item()
    inv = 1.0 / counts.clamp_min(1.0)
    inv[LINKER] *= linker_boost
    return inv * (3.0 / inv.sum())


def train_one_epoch(model, graphs, train_idx, optimizer, batch_size: int = 32,
                    w_bond: float = 0.5, w_smooth: float = 0.1,
                    w_recon: float = 0.1,
                    class_weights: Optional[torch.Tensor] = None) -> dict:
    model.train()
    total = {"loss": 0.0, "atom_loss": 0.0, "bond_loss": 0.0,
             "smooth_loss": 0.0, "recon_loss": 0.0,
             "atom_correct": 0, "n_atoms": 0}
    for b in iterate_minibatches(graphs, train_idx, batch_size, shuffle=True):
        optimizer.zero_grad()
        al, bl, _ = model(b["x"], b["edge_index"])
        y = b["y"]
        L_a = F.cross_entropy(al, y, weight=class_weights)
        L_b = bond_boundary_loss(bl, y, b["edge_index"])
        L_s = fragment_smoothness_loss(al, y, b["edge_index"])
        L_r = soft_reconstruction_loss(al, b["edge_index"], b["batch"], b["n_graphs"])
        loss = L_a + w_bond * L_b + w_smooth * L_s + w_recon * L_r
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        n = y.numel()
        total["loss"]        += loss.item() * n
        total["atom_loss"]   += L_a.item() * n
        total["bond_loss"]   += L_b.item() * n
        total["smooth_loss"] += L_s.item() * n
        total["recon_loss"]  += L_r.item() * n
        total["atom_correct"] += (al.argmax(-1) == y).sum().item()
        total["n_atoms"] += n
    n_atoms = max(total["n_atoms"], 1)
    return {"loss": total["loss"] / n_atoms,
            "atom_loss": total["atom_loss"] / n_atoms,
            "bond_loss": total["bond_loss"] / n_atoms,
            "smooth_loss": total["smooth_loss"] / n_atoms,
            "recon_loss": total["recon_loss"] / n_atoms,
            "atom_acc": total["atom_correct"] / n_atoms}


# --- decoding + metrics ----------------------------------------------------

def largest_cc_per_class(mol: Chem.Mol, labels: list[int]) -> list[int]:
    """Post-hoc constraint: keep biggest connected blob of each class,
    reassign every other atom to its nearest 'kept' neighbour."""
    n = mol.GetNumAtoms()
    adj = [[] for _ in range(n)]
    for b in mol.GetBonds():
        adj[b.GetBeginAtomIdx()].append(b.GetEndAtomIdx())
        adj[b.GetEndAtomIdx()].append(b.GetBeginAtomIdx())
    seen = [False] * n; ccs = []
    for i in range(n):
        if seen[i]:
            continue
        cls = labels[i]; comp = []; q = deque([i]); seen[i] = True
        while q:
            u = q.popleft(); comp.append(u)
            for v in adj[u]:
                if not seen[v] and labels[v] == cls:
                    seen[v] = True; q.append(v)
        ccs.append((cls, comp))
    keep = set()
    for cls in (WARHEAD, LINKER, E3):
        cands = [c for c in ccs if c[0] == cls]
        if cands:
            keep.update(max(cands, key=lambda c: len(c[1]))[1])
    if not keep:
        return list(labels)
    new = [-1] * n; q = deque()
    for a in keep:
        new[a] = labels[a]; q.append(a)
    while q:
        u = q.popleft()
        for v in adj[u]:
            if new[v] == -1:
                new[v] = new[u]; q.append(v)
    return new


def _macro_f1(y_true: list[int], y_pred: list[int], n_classes: int = 3) -> float:
    f1s = []
    for c in range(n_classes):
        tp = sum(1 for t, p in zip(y_true, y_pred) if t == c and p == c)
        fp = sum(1 for t, p in zip(y_true, y_pred) if t != c and p == c)
        fn = sum(1 for t, p in zip(y_true, y_pred) if t == c and p != c)
        prec = tp / max(tp + fp, 1); rec = tp / max(tp + fn, 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-9)
        f1s.append(f1)
    return sum(f1s) / n_classes


def _pr_auc(y_true: list[int], scores: list[float]) -> float:
    """Average precision from scratch (no sklearn dependency)."""
    if not y_true:
        return 0.0
    order = sorted(range(len(scores)), key=lambda i: -scores[i])
    P = sum(y_true)
    if P == 0:
        return 0.0
    tp = fp = 0; ap = 0.0; prev_recall = 0.0
    for k, i in enumerate(order, 1):
        if y_true[i]:
            tp += 1
        else:
            fp += 1
        prec = tp / (tp + fp); rec = tp / P
        ap += prec * (rec - prev_recall)
        prev_recall = rec
    return ap


@torch.no_grad()
def predict_labels(model, g: dict, constrain: bool = True) -> list[int]:
    """Return atom-class list for one graph."""
    b = collate([g])
    al, _, _ = model(b["x"], b["edge_index"])
    raw = al.argmax(-1).tolist()
    if not constrain:
        return raw
    mol = Chem.MolFromSmiles(g["smiles"])
    return largest_cc_per_class(mol, raw)


def evaluate_full(model, graphs: list[dict], indices: list[int]) -> dict:
    """Full metric bundle (raw + constrained) for a list of graph indices."""
    model.eval()
    y_true_atom, y_pred_raw, y_pred_cst = [], [], []
    y_true_bond, s_bond = [], []
    n_mol = 0; exact3_raw = exact3_cst = 0
    valid_raw = valid_cst = 0; reasm_raw = reasm_cst = 0
    with torch.no_grad():
        for idx in indices:
            g = graphs[idx]
            b = collate([g])
            al, bl, _ = model(b["x"], b["edge_index"])
            raw = al.argmax(-1).tolist()
            mol = Chem.MolFromSmiles(g["smiles"])
            cst = largest_cc_per_class(mol, raw)
            y = g["y"].tolist()
            y_true_atom += y; y_pred_raw += raw; y_pred_cst += cst
            # boundary
            src, dst = b["edge_index"][0].tolist(), b["edge_index"][1].tolist()
            bond_y = [int(y[i] != y[j]) for i, j in zip(src, dst)]
            bond_score = F.softmax(bl, dim=-1)[:, 1].tolist()
            y_true_bond += bond_y; s_bond += bond_score
            # molecule-level
            n_mol += 1
            ok_r, n_r = reassembles(mol, raw); ok_c, n_c = reassembles(mol, cst)
            exact3_raw += (n_r == 3); exact3_cst += (n_c == 3)
            valid_raw += ok_r; valid_cst += ok_c
            reasm_raw += ok_r; reasm_cst += ok_c
    return {
        "n_test_molecules": n_mol,
        "atom_acc_raw":  sum(1 for t, p in zip(y_true_atom, y_pred_raw) if t == p) / max(len(y_true_atom), 1),
        "atom_acc_cst":  sum(1 for t, p in zip(y_true_atom, y_pred_cst) if t == p) / max(len(y_true_atom), 1),
        "atom_macro_f1_raw": _macro_f1(y_true_atom, y_pred_raw),
        "atom_macro_f1_cst": _macro_f1(y_true_atom, y_pred_cst),
        "bond_boundary_pr_auc": _pr_auc(y_true_bond, s_bond),
        "exact_3_frag_raw": exact3_raw / max(n_mol, 1),
        "exact_3_frag_cst": exact3_cst / max(n_mol, 1),
        "reassembly_raw":  reasm_raw / max(n_mol, 1),
        "reassembly_cst":  reasm_cst / max(n_mol, 1),
    }

# =============================================================================
# helpers for the demo notebook
# =============================================================================

def draw_labeled(smiles: str, labels: list[int], title: str = "") -> "PIL.Image.Image":
    from PIL import Image
    mol = Chem.MolFromSmiles(smiles)
    atom_colors = {i: CLASS_COLORS[l] for i, l in enumerate(labels)}
    bond_colors = {}
    for b in mol.GetBonds():
        li, lj = labels[b.GetBeginAtomIdx()], labels[b.GetEndAtomIdx()]
        bond_colors[b.GetIdx()] = CLASS_COLORS[li] if li == lj else (0.15, 0.15, 0.15)
    d = rdMolDraw2D.MolDraw2DCairo(700, 500)
    d.drawOptions().highlightRadius = 0.35
    d.drawOptions().fillHighlights = True
    counts = Counter(labels)
    legend = f'{title}   W:{counts.get(0,0)}  L:{counts.get(1,0)}  E3:{counts.get(2,0)}'
    rdMolDraw2D.PrepareAndDrawMolecule(
        d, mol,
        highlightAtoms=list(atom_colors), highlightAtomColors=atom_colors,
        highlightBonds=list(bond_colors), highlightBondColors=bond_colors,
        legend=legend,
    )
    d.FinishDrawing()
    return Image.open(io.BytesIO(d.GetDrawingText()))


## 2. Load model

In [ ]:
import pickle, torch, random

blob = torch.load(PROCESSED / "graphs.pt", weights_only=False)
ckpt = torch.load(OUT / "gnn_models.pt", weights_only=False)
graphs = blob["graphs"]
splits = pickle.load(open(PROCESSED / "splits.pkl", "rb"))

family = "pretrained" if "pretrained" in ckpt["state_dicts"] else "scratch"
m = ProtacSegGNN(atom_dim=ckpt["atom_dim"], hidden=64, n_layers=4, dropout=0.0)
m.load_state_dict(ckpt["state_dicts"][family]["random"])
m.eval()
print("loaded", family, "/ random")


## 3. Paint 3 random-split test molecules

In [ ]:
random.seed(0)
picks = random.sample(splits["random"]["test"], 3)
for i, idx in enumerate(picks):
    g = graphs[idx]
    pred = predict_labels(m, g, constrain=True)
    img = draw_labeled(g["smiles"], pred, title=f"random-test #{i}")
    fp = OUT / "figures" / f"05_demo_{i}.png"
    img.save(fp)
    display(img)
    print("saved", fp)


## 4. Paint one hard OOD example

In [ ]:
for key, tag in [("fingerprint_ood", "fp_ood"), ("unseen_warhead", "unseen_wh")]:
    if key not in splits or not splits[key]["test"]:
        continue
    if key in ckpt["state_dicts"][family]:
        m2 = ProtacSegGNN(atom_dim=ckpt["atom_dim"], hidden=64, n_layers=4, dropout=0.0)
        m2.load_state_dict(ckpt["state_dicts"][family][key]); m2.eval()
    else:
        m2 = m
    g = graphs[splits[key]["test"][0]]
    pred = predict_labels(m2, g, constrain=True)
    img = draw_labeled(g["smiles"], pred, title=f"{tag} test")
    fp = OUT / "figures" / f"05_demo_{tag}.png"
    img.save(fp)
    display(img)
    print("saved", fp)

print("DONE — Notebook 05 complete.")
